In [ ]:
# Import libraries for the pipeline
import hashlib
import logging
import os
import sqlite3
import sys
from datetime import datetime

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# Configure paths and business rules
BASE_DIR = os.getcwd()
DB_PATH = os.path.join(BASE_DIR, "shop_oltp_p10.db")
LOG_PATH = os.path.join(BASE_DIR, "logs", "content_engagement.log")
QUARANTINE_PATH = os.path.join(BASE_DIR, "quarantine", "invalid_playback_lengths.csv")
ANALYTICS_DIR = os.path.join(BASE_DIR, "data", "analytics")

# Business rules
MIN_VALID_MINUTES = 0.01          # playback must be > 0
MAX_VALID_MINUTES = 720.0         # 12 hours; anything above is treated as a
                                   # sensor/logging error, not a real single
                                   # viewing session
VALID_STATUSES = {"COMPLETED", "REFUNDED", "PENDING", "CANCELLED"}

STATUS_NORMALIZATION_MAP = {
    "COMPLETED": "COMPLETED",
    "COMPLETD": "COMPLETED",
    "REFUNDED": "REFUNDED",
    "PENDING": "PENDING",
    "CANCELLED": "CANCELLED",
    "ERROR": "ERROR",
    "N/A": "UNKNOWN",
    "UNKNOWN_STATUS": "UNKNOWN",
}

GARBAGE_GENRE_TOKENS = {
    "", "unknown", "???", "#n/a", "n/a", "na", "null", "nulll",
    "<blank>", "12345",
}

# Corrects known fat-finger / duplicate-encoding typos after normalization
GENRE_TYPO_FIXES = {
    "ACTIONN": "ACTION",
}

In [ ]:
# Set up file and console logging

def setup_logging() -> logging.Logger:
    os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
    logger = logging.getLogger("content_engagement_pipeline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    )

    file_handler = logging.FileHandler(LOG_PATH, mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger

In [ ]:
# Extract raw watch events from SQLite

def extract(db_path: str, logger: logging.Logger) -> pd.DataFrame:
    logger.info("EXTRACT | Connecting to source ledger: %s", db_path)
    conn = sqlite3.connect(db_path)
    try:
        df = pd.read_sql("SELECT * FROM orders", conn)
    finally:
        conn.close()
    logger.info("EXTRACT | Pulled %d raw watch-event records", len(df))
    return df

In [ ]:
# Normalize genres, statuses, dates, and user identifiers

def normalize_genre(raw_genre) -> str:
    if raw_genre is None:
        return "UNKNOWN"
    g = str(raw_genre).strip()

    if g.lower() in GARBAGE_GENRE_TOKENS or g.lower().startswith("genre_error"):
        return "UNKNOWN"

    # Collapse separators (-, _, !, multiple spaces) to a single space so
    # "K-Drama", "K_Drama", "K Drama" and "Sci_Fi!!" all converge.
    cleaned = "".join(ch if ch.isalnum() else " " for ch in g)
    cleaned = " ".join(cleaned.split()).upper()

    if cleaned == "":
        return "UNKNOWN"

    return GENRE_TYPO_FIXES.get(cleaned, cleaned)


def normalize_status(raw_status) -> str:
    if raw_status is None or pd.isna(raw_status):
        return "UNKNOWN"
    s = str(raw_status).strip().upper()
    if s == "":
        return "UNKNOWN"
    return STATUS_NORMALIZATION_MAP.get(s, s if s in VALID_STATUSES else "UNKNOWN")


def normalize_date(raw_date):
    if raw_date is None:
        return pd.NaT
    s = str(raw_date).strip()
    if s == "" or s.lower() == "not_a_date":
        return pd.NaT
    parsed = pd.to_datetime(s, errors="coerce")
    return parsed


def anonymize_user_id(user_id: str) -> str:
    """SHA-256 hash of the account identifier for PII-safe analytics."""
    return hashlib.sha256(user_id.strip().encode("utf-8")).hexdigest()

In [ ]:
# Validate records and quarantine invalid playback data

def validate_and_clean(df: pd.DataFrame, logger: logging.Logger):
    logger.info("VALIDATE | Beginning field-level validation and normalization")

    work = df.copy()
    work["video_genre"] = work["video_genre"].apply(normalize_genre)
    work["order_status"] = work["order_status"].apply(normalize_status)
    work["order_date_parsed"] = work["order_date"].apply(normalize_date)

    # Flag reasons a record is invalid / must be quarantined
    reasons = []
    for _, row in work.iterrows():
        row_reasons = []
        duration = row["playback_duration_minutes"]

        if pd.isna(duration):
            row_reasons.append("missing_playback_duration")
        elif duration <= MIN_VALID_MINUTES:
            row_reasons.append("non_positive_playback_duration")
        elif duration > MAX_VALID_MINUTES:
            row_reasons.append("implausible_playback_duration")

        if pd.isna(row["user_account_id"]) or str(row["user_account_id"]).strip() == "":
            row_reasons.append("missing_user_account_id")

        if pd.isna(row["order_date_parsed"]):
            row_reasons.append("invalid_or_missing_order_date")

        reasons.append(";".join(row_reasons) if row_reasons else "")

    work["quarantine_reason"] = reasons
    is_bad = work["quarantine_reason"] != ""

    quarantined = work[is_bad].copy()
    clean = work[~is_bad].copy()

    logger.info(
        "VALIDATE | %d records passed validation, %d records quarantined",
        len(clean),
        len(quarantined),
    )

    return clean, quarantined


def quarantine_bad_records(quarantined: pd.DataFrame, logger: logging.Logger) -> None:
    os.makedirs(os.path.dirname(QUARANTINE_PATH), exist_ok=True)
    cols = [
        "order_id",
        "user_account_id",
        "video_genre",
        "playback_duration_minutes",
        "order_status",
        "order_date",
        "quarantine_reason",
    ]
    quarantined[cols].to_csv(QUARANTINE_PATH, index=False)
    logger.info(
        "QUARANTINE | Wrote %d isolated records to %s",
        len(quarantined),
        QUARANTINE_PATH,
    )

In [ ]:
# Hash user identifiers before analytics processing

def anonymize(clean: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    logger.info("ANONYMIZE | Hashing user_account_id with SHA-256 for %d records", len(clean))
    clean = clean.copy()
    clean["user_account_id_hash"] = clean["user_account_id"].apply(anonymize_user_id)
    return clean

In [ ]:
# Aggregate daily streaming duration and genre rankings

def aggregate_daily_genre_engagement(clean: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    logger.info("TRANSFORM | Aggregating total daily streaming duration per genre")

    clean["watch_date"] = clean["order_date_parsed"].dt.date.astype(str)
    clean["duration_hours"] = clean["playback_duration_minutes"] / 60.0

    agg = (
        clean.groupby(["watch_date", "video_genre"], as_index=False)["duration_hours"]
        .sum()
        .rename(columns={"duration_hours": "total_duration_hours"})
    )
    agg["total_duration_hours"] = agg["total_duration_hours"].round(2)

    # Rank genres within each day by total engagement (1 = highest)
    agg["daily_genre_rank"] = (
        agg.groupby("watch_date")["total_duration_hours"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    agg = agg.sort_values(["watch_date", "daily_genre_rank"]).reset_index(drop=True)
    logger.info(
        "TRANSFORM | Produced %d daily-genre engagement rows across %d unique days",
        len(agg),
        agg["watch_date"].nunique(),
    )
    return agg

In [ ]:
# Write the aggregated results as Hive-partitioned Parquet

def load_partitioned_parquet(agg: pd.DataFrame, logger: logging.Logger) -> None:
    os.makedirs(ANALYTICS_DIR, exist_ok=True)
    table = pa.Table.from_pandas(agg, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=os.path.join(ANALYTICS_DIR, "genre_engagement_rankings.parquet"),
        partition_cols=["video_genre"],
    )
    logger.info(
        "LOAD | Wrote Hive-partitioned dataset (partitioned by video_genre) to %s",
        os.path.join(ANALYTICS_DIR, "genre_engagement_rankings.parquet"),
    )

In [ ]:
# Run each pipeline stage and log the complete execution

def run_pipeline(db_path: str) -> pd.DataFrame:
    logger = setup_logging()
    started = datetime.now()
    logger.info("PIPELINE | Run started")

    try:
        raw = extract(db_path, logger)
        clean, quarantined = validate_and_clean(raw, logger)
        quarantine_bad_records(quarantined, logger)
        clean = anonymize(clean, logger)
        daily_genre_agg = aggregate_daily_genre_engagement(clean, logger)
        load_partitioned_parquet(daily_genre_agg, logger)

        elapsed = (datetime.now() - started).total_seconds()
        logger.info("PIPELINE | Run completed successfully in %.2fs", elapsed)
        return daily_genre_agg
    except Exception:
        logger.exception("PIPELINE | Run failed with an unhandled exception")
        raise

In [ ]:
# Execute the pipeline and preview the aggregated results
result_df = run_pipeline(DB_PATH)
result_df.head(10)

2026-08-20 03:17:23 | INFO     | PIPELINE | Run started
2026-08-20 03:17:23 | INFO     | EXTRACT | Connecting to source ledger: c:\Users\Mary\Documents\GitHub\media-streaming-data-pipeline\shop_oltp_p10.db
2026-08-20 03:17:23 | INFO     | EXTRACT | Pulled 4000 raw watch-event records
2026-08-20 03:17:23 | INFO     | VALIDATE | Beginning field-level validation and normalization
2026-08-20 03:17:25 | INFO     | VALIDATE | 3797 records passed validation, 203 records quarantined
2026-08-20 03:17:25 | INFO     | QUARANTINE | Wrote 203 isolated records to c:\Users\Mary\Documents\GitHub\media-streaming-data-pipeline\quarantine\invalid_playback_lengths.csv
2026-08-20 03:17:25 | INFO     | ANONYMIZE | Hashing user_account_id with SHA-256 for 3797 records
2026-08-20 03:17:25 | INFO     | TRANSFORM | Aggregating total daily streaming duration per genre
2026-08-20 03:17:25 | INFO     | TRANSFORM | Produced 3483 daily-genre engagement rows across 1239 unique days
2026-08-20 03:17:26 | INFO     | LO

,watch_date,video_genre,total_duration_hours,daily_genre_rank
0,1970-01-01,ANIME,10.90,1
1,1970-01-01,ROMANCE,7.35,2
2,1970-01-01,DRAMA,6.53,3
3,1970-01-01,HORROR,2.36,4
4,1970-01-01,ACTION,0.78,5
5,2023-01-01,TRUE CRIME,0.33,1
6,2023-01-02,REALITY TV,6.82,1
7,2023-01-02,ANIME,5.31,2
8,2023-01-02,ANIMATION,0.64,3
9,2023-01-03,FANTASY,5.98,1


In [ ]:
# Review quarantine reasons and counts
quarantine_df = pd.read_csv(QUARANTINE_PATH)
print(f"Quarantined records: {len(quarantine_df)}")
quarantine_df["quarantine_reason"].value_counts()

Quarantined records: 203


quarantine_reason
non_positive_playback_duration                                  61
invalid_or_missing_order_date                                   57
missing_user_account_id                                         37
missing_playback_duration                                       20
implausible_playback_duration                                   16
non_positive_playback_duration;invalid_or_missing_order_date     4
implausible_playback_duration;invalid_or_missing_order_date      2
missing_playback_duration;missing_user_account_id                2
missing_user_account_id;invalid_or_missing_order_date            1
non_positive_playback_duration;missing_user_account_id           1
implausible_playback_duration;missing_user_account_id            1
missing_playback_duration;invalid_or_missing_order_date          1
Name: count, dtype: int64

In [ ]:
# Read the Parquet output and verify summary dimensions
readback = pq.read_table(
    os.path.join(ANALYTICS_DIR, "genre_engagement_rankings.parquet")
).to_pandas()

print(f"Rows: {len(readback)}  |  Genres: {readback['video_genre'].nunique()}  |  Days: {readback['watch_date'].nunique()}")
readback.sort_values(["watch_date", "daily_genre_rank"]).head(10)

Rows: 3483  |  Genres: 17  |  Days: 1239


,watch_date,total_duration_hours,daily_genre_rank,video_genre
436,1970-01-01,10.90,1,ANIME
2298,1970-01-01,7.35,2,ROMANCE
1030,1970-01-01,6.53,3,DRAMA
1477,1970-01-01,2.36,4,HORROR
0,1970-01-01,0.78,5,ACTION
3187,2023-01-01,0.33,1,TRUE CRIME
2105,2023-01-02,6.82,1,REALITY TV
437,2023-01-02,5.31,2,ANIME
215,2023-01-02,0.64,3,ANIMATION
1263,2023-01-03,5.98,1,FANTASY
